In [1]:
import numpy as np                                          
from scipy.stats import norm, bernoulli, norm as scipy_norm 
import pandas as pd                                         
from tqdm.notebook import tqdm                              
%load_ext autoreload
%autoreload 2                                               

from utils.inference import (
    train_sampling_rule,          # fits XGBoost to predict LLM squared error from demo features
    sampling_rule_predict,        # applies fitted rule to get per-observation error predictions
    perspective_driven_inference, # PDI estimator + bootstrap CI
)
from ppi_py import ppi_mean_pointestimate, ppi_mean_ci      # PPI++ point estimate and CI

In [2]:
alpha        = 0.05   # significance level → 95% confidence intervals
burnin_steps = 100     # first 50 obs collected uniformly to bootstrap the sampling rule
n_batches    = 3      # number of adaptive batches after burn-in
n_human      = 200    # total human annotation budget (burn-in + adaptive batches)
N            = None   # placeholder; set after data loading
random_state = 42     # global random seed for numpy and XGBoost
n_trials     = 20     # number of independent trial repetitions per model
tau          = 0.1    # smoothing weight: mixes adaptive probs with uniform within each batch
q_power      = 1.0    # exponent on predicted error when computing sampling probs

offensive_threshold = 2  # binary label: offensive = rating >= 4 on a 1–5 scale

demo_cols = ['gender', 'race', 'age', 'education']

# Non-binary absent from this dataset's LLM predictions
demographic_groups = {
    # gender
    'Woman':            lambda df: df['gender'] == 'Woman',
    'Man':              lambda df: df['gender'] == 'Man',
    # race
    'White':            lambda df: df['race'] == 'White',
    'Black/Afr. Am.':   lambda df: df['race'] == 'Black or African American',
    'Hispanic/Latino':  lambda df: df['race'] == 'Hispanic or Latino',
    'Asian':            lambda df: df['race'] == 'Asian',
    # age (broad buckets)
    'Age 18-34':        lambda df: df['age'].isin(['18-24', '25-29', '30-34']),
    'Age 35-49':        lambda df: df['age'].isin(['35-39', '40-44', '45-49']),
    'Age 50+':          lambda df: df['age'].isin(['50-54', '54-59', '60-64', '>65']),
    # education
    'College degree':   lambda df: df['education'] == 'College degree',
    'HS diploma':       lambda df: df['education'] == 'High school diploma or equivalent',
    'Grad. degree':     lambda df: df['education'] == 'Graduate degree',
}

def mean_estimator(y, weights):
    y, weights = y[~np.isnan(y)], weights[~np.isnan(y)]
    return np.sum(y * weights) / np.sum(weights)

group_names = list(demographic_groups.keys())


In [3]:
all_models = [
    'gpt-5.2',
    'claude-sonnet-4.6',
    'claude-opus-4.6',
    'gemini-3.1-pro',
    'claude-haiku-4.5',
    'llama-3-8b',
    'mistral-large-2512',
    'gpt-oss-120b',
]

# only evaluate this model, but filter data to the intersection of all 8
models = ['gpt-5.2']

model_cols = {
    m: {
        'zs': f'{m} (zero shot prompting)',
        'fs': f'{m} (few shot)',
        'pp': f'{m} (persona prompt)',
    }
    for m in all_models
}

# keep only rows where ALL 8 models have predictions, dropping Non-binary
required_cols = [c for m in all_models for c in model_cols[m].values()]
data_mm = (
    pd.read_csv('data/raw_data_llm_ofensiveness.csv')
      .query("gender != 'Non-binary'")
      .dropna(subset=required_cols)
      .sample(frac=1, random_state=random_state)
      .reset_index(drop=True)
      .copy()
)
data_mm['human_true'] = (data_mm['offensiveness'] >= offensive_threshold).astype(float)
N_mm = len(data_mm)

demo_features_mm = pd.get_dummies(data_mm[demo_cols]).astype(float).values
group_masks_mm   = [mask_fn(data_mm).values for mask_fn in demographic_groups.values()]
true_thetas_mm   = np.array([data_mm.loc[m, 'human_true'].mean() for m in group_masks_mm])
group_sizes_mm   = np.array([m.sum() for m in group_masks_mm])
expected_n_mm    = n_human * group_sizes_mm / N_mm
var_human_mm     = true_thetas_mm * (1 - true_thetas_mm) / expected_n_mm

dimensions = {
    'Gender':    ['Woman', 'Man'],
    'Race':      ['White', 'Black/Afr. Am.', 'Hispanic/Latino', 'Asian'],
    'Age':       ['Age 18-34', 'Age 35-49', 'Age 50+'],
    'Education': ['College degree', 'HS diploma', 'Grad. degree'],
}
dim_idx = {dim: [group_names.index(n) for n in names] for dim, names in dimensions.items()}

gender_names_mm = dimensions['Gender']
gender_idx_mm   = dim_idx['Gender']

print(f'Multi-model dataset: N={N_mm}')
print(f'  Human offensive rate: {data_mm["human_true"].mean():.3f}')
print(f'  Gender distribution: {data_mm["gender"].value_counts().to_dict()}')


Multi-model dataset: N=801
  Human offensive rate: 0.320
  Gender distribution: {'Woman': 568, 'Man': 233}


In [4]:
# Debug: per-group LLM error rate, group size, and base rate for gpt-5.2
_cols = model_cols['gpt-5.2']
_zs = (data_mm[_cols['zs']] >= offensive_threshold).astype(float).to_numpy()
_fs = (data_mm[_cols['fs']] >= offensive_threshold).astype(float).to_numpy()
_pp = (data_mm[_cols['pp']] >= offensive_threshold).astype(float).to_numpy()
_h  = data_mm['human_true'].values

print(f"{'Group':<22} {'Size':>6} {'Base rate':>10} {'ZS err':>8} {'FS err':>8} {'PP err':>8}")
print('-' * 70)
_group_errors = {'zero shot': [], 'few shot': [], 'persona': []}
for name, mask in zip(group_names, group_masks_mm):
    size = mask.sum()
    if size == 0:
        print(f"{name:<22} {'(empty)':>6}")
        continue
    base_rate = _h[mask].mean()
    zs_err = (_zs[mask] != _h[mask]).mean()
    fs_err = (_fs[mask] != _h[mask]).mean()
    pp_err = (_pp[mask] != _h[mask]).mean()
    _group_errors['zero shot'].append((zs_err, name))
    _group_errors['few shot'].append((fs_err, name))
    _group_errors['persona'].append((pp_err, name))
    print(f"{name:<22} {size:>6} {base_rate:>10.3f} {zs_err:>8.3f} {fs_err:>8.3f} {pp_err:>8.3f}")

# Overall best prompting by mean error across groups
_overall_err = {v: np.mean([e for e, _ in errs]) for v, errs in _group_errors.items()}
print()
print("Overall mean error across groups:")
for vname, err in _overall_err.items():
    print(f"  {vname:<12}: {err:.3f}")
_best = min(_overall_err, key=lambda v: _overall_err[v])
print(f"  → best prompting: {_best}")

# Group with largest error for the best prompting
_worst_err, _worst_group = max(_group_errors[_best])
print(f"  → largest error group for '{_best}': {_worst_group}  (err={_worst_err:.3f})")


Group                    Size  Base rate   ZS err   FS err   PP err
----------------------------------------------------------------------
Woman                     568      0.299    0.340    0.338    0.336
Man                       233      0.369    0.266    0.249    0.245
White                     649      0.328    0.320    0.307    0.311
Black/Afr. Am.             35      0.343    0.200    0.200    0.229
Hispanic/Latino        (empty)
Asian                      84      0.357    0.310    0.393    0.286
Age 18-34                 353      0.329    0.295    0.329    0.280
Age 35-49                 364      0.349    0.335    0.286    0.321
Age 50+                    84      0.155    0.345    0.357    0.381
College degree            408      0.370    0.321    0.321    0.306
HS diploma                280      0.318    0.307    0.300    0.293
Grad. degree               45      0.067    0.378    0.378    0.422

Overall mean error across groups:
  zero shot   : 0.311
  few shot    : 0.314
  p

In [5]:
def run_model_trials(Hhat_best, Hhat_zs, Hhat_fs, Hhat_pp, data, demo_features,
                     group_masks, true_thetas):
    """Run PDI/PPI/LLM-only trials for one model.

    Hhat_best : LLM predictions used for PPI/PDI (best-coverage variant, chosen in the run cell)
    Hhat_zs/fs/pp : all three prompting variants, shown as LLM-only baselines in the table
    """
    n_groups = len(group_masks)
    N_loc    = len(data)

    methods   = ['ZS LLM', 'FS LLM', 'PP LLM', 'PPI', 'PDI']
    ests      = {m: np.full((n_trials, n_groups), np.nan) for m in methods}   # point estimates
    covered   = {m: np.full((n_trials, n_groups), np.nan) for m in methods}   # 1 if CI covers true theta
    vars_est  = {m: np.full((n_trials, n_groups), np.nan) for m in methods}   # within-trial variance
    # per-trial actual baseline variance for PDI: theta*(1-theta) / n_actually_sampled_in_group
    vars_human_actual = np.full((n_trials, n_groups), np.nan)

    batch_size_loc = (N_loc - burnin_steps) // n_batches
    z_crit = scipy_norm.ppf(1 - alpha / 2)

    def compute_q_loc(rule, features):
        return np.clip(sampling_rule_predict(rule, features), 1e-8, None) ** q_power

    for trial in range(n_trials):
        np.random.seed(random_state + trial * 13)

        ppi_idx      = np.random.choice(N_loc, size=n_human, replace=False)
        ppi_selected = np.zeros(N_loc, dtype=bool)
        ppi_selected[ppi_idx] = True

        H  = np.full(N_loc, np.nan)
        SP = np.zeros(N_loc)
        SD = np.zeros(N_loc)

        H[:burnin_steps]  = data['human_true'].values[:burnin_steps]
        SP[:burnin_steps] = 1.0
        SD[:burnin_steps] = 1.0

        sampling_rule = train_sampling_rule(
            demo_features[:burnin_steps],
            (H[:burnin_steps] - Hhat_best[:burnin_steps]) ** 2,
            seed=random_state)
        q = compute_q_loc(sampling_rule, demo_features)

        for b in range(n_batches):
            batch_inds = (
                np.arange(burnin_steps + b * batch_size_loc,
                          burnin_steps + (b + 1) * batch_size_loc)
                if b < n_batches - 1
                else np.arange(burnin_steps + b * batch_size_loc, N_loc)
            )
            remaining = n_human - int(SD.sum())
            if remaining <= 0:
                break
            target = min(round(remaining / (n_batches - b)), remaining, len(batch_inds))

            p = q[batch_inds];  p = p / p.sum()
            p = (1 - tau) * p + tau / len(batch_inds);  p = p / p.sum()

            sel = np.random.choice(len(batch_inds), size=target, replace=False, p=p)

            H[batch_inds[sel]] = data['human_true'].values[batch_inds[sel]]
            SD[batch_inds] = 0.0;  SD[batch_inds[sel]] = 1.0
            SP[batch_inds] = np.clip(target * p, 1e-4, 1.0)

            if b < n_batches - 1:
                lab = np.where(~np.isnan(H))[0]
                sampling_rule = train_sampling_rule(
                    demo_features[lab], (H[lab] - Hhat_best[lab]) ** 2,
                    seed=random_state)
                q = compute_q_loc(sampling_rule, demo_features)

        # Debug header for this trial
        print(f"\n--- Trial {trial+1}/{n_trials} ---")
        print(f"  {'Group':<22} {'PDI samples':>12} {'PPI samples':>12}")
        print(f"  {'-'*48}")

        for g, (mask, true_theta) in enumerate(zip(group_masks, true_thetas)):
            llm_mask    = mask & ppi_selected
            g_sampled   = mask & ppi_selected
            g_unsampled = mask & ~ppi_selected

            # Debug: sample counts per group
            n_pdi = int(SD[mask].sum())
            n_ppi = int(ppi_selected[mask].sum())
            print(f"  {group_names[g]:<22} {n_pdi:>12} {n_ppi:>12}")

            # LLM-only baselines
            for lm_key, Hhat_lm in [('ZS LLM', Hhat_zs),
                                     ('FS LLM', Hhat_fs),
                                     ('PP LLM', Hhat_pp)]:
                if llm_mask.sum() >= 2:
                    lm_est = Hhat_lm[llm_mask].mean()
                    se2 = max(lm_est * (1 - lm_est), 1e-6) / llm_mask.sum()
                    se  = np.sqrt(se2)
                    ests[lm_key][trial, g]     = lm_est
                    covered[lm_key][trial, g]  = int(lm_est - z_crit * se <= true_theta <= lm_est + z_crit * se)
                    vars_est[lm_key][trial, g] = se2

            # PPI++
            if g_sampled.sum() >= 2 and g_unsampled.sum() >= 1:
                Y_lab      = data['human_true'].values[g_sampled]
                Yhat_lab   = Hhat_best[g_sampled]
                Yhat_unlab = Hhat_best[g_unsampled]
                est    = ppi_mean_pointestimate(Y_lab, Yhat_lab, Yhat_unlab, lam=None)
                lb, ub = ppi_mean_ci(Y_lab, Yhat_lab, Yhat_unlab, alpha=alpha, lam=None)
                ests['PPI'][trial, g]     = est
                covered['PPI'][trial, g]  = int(lb <= true_theta <= ub)
                vars_est['PPI'][trial, g] = ((ub - lb) / (2 * z_crit)) ** 2

            # PDI
            if SD[mask].sum() >= 2:
                n_g_actual = SD[mask].sum()
                vars_human_actual[trial, g] = true_theta * (1 - true_theta) / n_g_actual
                est, (lb, ub), pdi_std = perspective_driven_inference(
                    mean_estimator,
                    Y=H[mask], Yhat=Hhat_best[mask],
                    sampling_probs=SP[mask], sampling_decisions=SD[mask],
                    alpha=alpha, lam=None,
                    n_resamples=100, n_resamples_lam=20)
                ests['PDI'][trial, g]     = est
                covered['PDI'][trial, g]  = int(lb <= true_theta <= ub)
                vars_est['PDI'][trial, g] = pdi_std ** 2

    return ests, covered, vars_est, vars_human_actual


In [6]:
model_results_mm = {}

for model in tqdm(models, desc='Models'):
    cols = model_cols[model]

    missing = [c for c in cols.values() if c not in data_mm.columns]
    if missing:
        print(f'Skipping {model}: missing {missing}')
        continue

    Hhat_zs = (data_mm[cols['zs']] >= offensive_threshold).astype(float).to_numpy()
    Hhat_fs = (data_mm[cols['fs']] >= offensive_threshold).astype(float).to_numpy()
    Hhat_pp = (data_mm[cols['pp']] >= offensive_threshold).astype(float).to_numpy()
    human   = data_mm['human_true'].values

    # Select best prompting variant by overall accuracy before running trials
    accs = {
        'zero shot': (Hhat_zs == human).mean(),
        'few shot':  (Hhat_fs == human).mean(),
        'persona':   (Hhat_pp == human).mean(),
    }
    print(f'\n{model} prompting accuracy (higher = better):')
    for vname, acc in accs.items():
        print(f'  {vname:<12}: {acc:.3f}')
    best_variant = max(accs, key=lambda v: accs[v])
    print(f'  → using: {best_variant}')

    Hhat_best = {'zero shot': Hhat_zs, 'few shot': Hhat_fs, 'persona': Hhat_pp}[best_variant]
    e, c, v, vha = run_model_trials(
        Hhat_best, Hhat_zs, Hhat_fs, Hhat_pp,
        data_mm, demo_features_mm, group_masks_mm, true_thetas_mm)

    model_results_mm[model] = {
        'ests':              e,
        'covered':           c,
        'vars_est':          v,
        'vars_human_actual': vha,
        'true_thetas':       true_thetas_mm,
        'best_variant':      best_variant,
        'var_human':         var_human_mm,
        'N_mm':              N_mm,
        'llm_err_zs':  (Hhat_zs != human).mean(),
        'llm_err_fs':  (Hhat_fs != human).mean(),
        'llm_err_pp':  (Hhat_pp != human).mean(),
    }

print('\nDone.')


Models:   0%|          | 0/1 [00:00<?, ?it/s]


gpt-5.2 prompting accuracy (higher = better):
  zero shot   : 0.682
  few shot    : 0.688
  persona     : 0.690
  → using: persona

--- Trial 1/20 ---
  Group                   PDI samples  PPI samples
  ------------------------------------------------
  Woman                           159          157
  Man                              41           43
  White                           167          165
  Black/Afr. Am.                    7            9
  Hispanic/Latino                   0            0
  Asian                            15           16
  Age 18-34                        79           83
  Age 35-49                        96           93
  Age 50+                          25           24
  College degree                   99          103
  HS diploma                       72           67
  Grad. degree                     11           11

--- Trial 2/20 ---
  Group                   PDI samples  PPI samples
  ------------------------------------------------
  Woman     

In [7]:
mw  = 25
sep = '=' * (mw + 26)
sub = '-' * (mw + 26)
ci_level_mm = int((1 - alpha) * 100)

def print_table_block(header, idx_list, ests_m, cov_m, true_thetas,
                      best_variant, zs_label, fs_label, pp_label, indent=''):
    """Print one table block (dimension average or single subgroup).

    Avg Delta = mean over trials and groups of |estimate - true_theta| * 100  (percentage points).
    """
    def avg_delta(key):
        return np.nanmean([
            np.nanmean(np.abs(ests_m[key][:, i] - true_thetas[i])) * 100
            for i in idx_list
        ])

    rows = [
        (zs_label, 'ZS LLM'),
        (fs_label, 'FS LLM'),
        (pp_label, 'PP LLM'),
        (f'PPI  [★ {best_variant}]', 'PPI'),
        (f'PDI  [★ {best_variant}]', 'PDI'),
    ]

    print(f'{indent}{header}')
    print(f'{indent}{sep}')
    print(f'{indent}{"Method":<{mw}} {"Avg Coverage":>12} {"Avg Delta (%pt)":>15}')
    print(f'{indent}{sub}')
    for label, key in rows:
        cov_avg = np.nanmean([np.nanmean(cov_m[key][:, i]) for i in idx_list])
        delta   = avg_delta(key)
        delta_str = f'{delta:>14.2f}' if not np.isnan(delta) else f"{'nan':>15}"
        print(f'{indent}{label:<{mw}} {cov_avg:>12.3f} {delta_str}')
    print(f'{indent}{sep}')

for model in models:
    if model not in model_results_mm:
        continue
    res          = model_results_mm[model]
    ests_m       = res['ests']
    cov_m        = res['covered']
    true_thetas  = res['true_thetas']
    best_variant = res['best_variant']

    zs_label = f'{model} (zero shot)' + (' ★' if best_variant == 'zero shot' else '')
    fs_label = f'{model} (few shot)'  + (' ★' if best_variant == 'few shot'  else '')
    pp_label = f'{model} (persona)'   + (' ★' if best_variant == 'persona'   else '')

    print(f'Coverage & Avg Delta  (target {ci_level_mm}%,  n_human={n_human},  {n_trials} trials,  N={res["N_mm"]})')
    print(f'  Avg Delta = mean |estimate - true theta| in percentage points  (lower = better)')
    print()

    for dim_name, names in dimensions.items():
        idx = dim_idx[dim_name]
        idx_arr = np.array(idx)

        print_table_block(
            f'Averaged across {dim_name}: {names}',
            idx_arr, ests_m, cov_m, true_thetas, best_variant,
            zs_label, fs_label, pp_label)
        print()

        for name, i in zip(names, idx):
            print_table_block(
                f'  {dim_name}: {name}',
                np.array([i]), ests_m, cov_m, true_thetas, best_variant,
                zs_label, fs_label, pp_label, indent='  ')
            print()


Coverage & Avg Delta  (target 95%,  n_human=200,  20 trials,  N=801)
  Avg Delta = mean |estimate - true theta| in percentage points  (lower = better)

Averaged across Gender: ['Woman', 'Man']
Method                    Avg Coverage Avg Delta (%pt)
---------------------------------------------------
gpt-5.2 (zero shot)              0.300          15.53
gpt-5.2 (few shot)               0.450           9.48
gpt-5.2 (persona) ★              0.350          14.68
PPI  [★ persona]                 0.925           3.44
PDI  [★ persona]                 0.875           7.51

    Gender: Woman
  Method                    Avg Coverage Avg Delta (%pt)
  ---------------------------------------------------
  gpt-5.2 (zero shot)              0.000          20.18
  gpt-5.2 (few shot)               0.050          12.61
  gpt-5.2 (persona) ★              0.000          19.48
  PPI  [★ persona]                 0.900           3.06
  PDI  [★ persona]                 0.900           3.88

    Gender: Man
  M